In [1]:
import os
import time
import datetime
# Correctly import simulator AND the constants needed for the test output
from simulator.core import (
    GreenhouseSimulator,
    ControlState,
    MIN_FAN_ON_DURATION, # <-- IMPORT CONSTANT
    MIN_FAN_OFF_DURATION, # <-- IMPORT CONSTANT
    # Import others if you add tests for them:
    # MIN_AC_ON_DURATION,
    # MIN_AC_OFF_DURATION,
    # MIN_VENT_ON_DURATION,
    # MIN_VENT_OFF_DURATION,
    # MIN_IRRIGATION_ON_DURATION,
    # MIN_IRRIGATION_OFF_DURATION,
)

# --- Configuration for the test ---
CITY_CONFIG_DIR = "simulator/city_configs" # Relative to your project root
city_to_test = "oslo" # or "riyadh"
city_config_path = os.path.join(CITY_CONFIG_DIR, f"{city_to_test}.yaml")

dt_minutes_per_step = 5

# --- Initialize Simulator ---
print(f"Initializing simulator for Cooldown Test ({city_to_test.capitalize()})...")
simulator = GreenhouseSimulator(
    city_config_path=city_config_path,
    start_year=2025, start_month=7, start_day_of_month=1, start_hour=6
)
print(f"Started at: {simulator.current_datetime.isoformat()}")
print("-" * 30)

# Test Fan Cooldown
print("\n--- Testing FAN Cooldown ---")
print(f"Initial Fan State: {simulator.actual_fan_on}")

# Request to turn FAN ON
print("Requesting FAN ON...")
controls_fan_on = ControlState(fan=True)
simulator.step(dt_minutes_per_step, controls_fan_on)
print(f"Step 1 ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should be ON

# Immediately request to turn FAN OFF (should be blocked by MIN_FAN_ON_DURATION)
print("Immediately Requesting FAN OFF (expecting it to stay ON due to cooldown)...")
controls_fan_off = ControlState(fan=False)
simulator.step(dt_minutes_per_step, controls_fan_off)
print(f"Step 2 ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should still be ON

# Run simulator for a bit longer than MIN_FAN_ON_DURATION (e.g., 3 steps * 5min = 15 min passes)
# Use the imported constant directly in the print statement
print(f"Running for a few steps to pass MIN_FAN_ON_DURATION ({MIN_FAN_ON_DURATION})...")
for i in range(3): # 3 steps * 5 min = 15 min. MIN_FAN_ON_DURATION is 15 min.
    # Keep requesting OFF, the cooldown should keep overriding until it expires
    simulator.step(dt_minutes_per_step, controls_fan_off)
    print(f"Step {3+i} ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should stay ON for steps 3, 4, 5

# Now request to turn FAN OFF again (cooldown should have expired after step 5)
print("Requesting FAN OFF again (should work now)...")
simulator.step(dt_minutes_per_step, controls_fan_off)
print(f"Step {3+3} ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should be OFF now

# Immediately request to turn FAN ON (should be blocked by MIN_FAN_OFF_DURATION)
# Use the imported constant directly
print(f"Immediately Requesting FAN ON (expecting it to stay OFF - MIN_FAN_OFF_DURATION is {MIN_FAN_OFF_DURATION})...")
simulator.step(dt_minutes_per_step, controls_fan_on) # Request ON
print(f"Step {3+3+1} ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should still be OFF

# Run a couple more steps to pass MIN_FAN_OFF_DURATION (10 min = 2 steps)
print(f"Running 2 more steps to pass MIN_FAN_OFF_DURATION...")
simulator.step(dt_minutes_per_step, controls_fan_on) # Request ON
print(f"Step {3+3+2} ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should still be OFF
simulator.step(dt_minutes_per_step, controls_fan_on) # Request ON
print(f"Step {3+3+3} ({simulator.current_datetime.isoformat()}) - Actual Fan State: {simulator.actual_fan_on}") # Should turn ON now

# You can add similar focused tests for AC, Vent, and Irrigation below,
# importing their specific MIN_... constants as needed.
# --- Testing AC Cooldown ---
# print("\n--- Testing AC Cooldown ---")
# ...

print("\nCooldown Test Script Finished.")

Initializing simulator for Cooldown Test (Oslo)...
Started at: 2025-07-01T06:00:00
------------------------------

--- Testing FAN Cooldown ---
Initial Fan State: False
Requesting FAN ON...
Step 1 (2025-07-01T06:05:00) - Actual Fan State: True
Immediately Requesting FAN OFF (expecting it to stay ON due to cooldown)...
Step 2 (2025-07-01T06:10:00) - Actual Fan State: True
Running for a few steps to pass MIN_FAN_ON_DURATION (0:15:00)...
Step 3 (2025-07-01T06:15:00) - Actual Fan State: True
Step 4 (2025-07-01T06:20:00) - Actual Fan State: False
Step 5 (2025-07-01T06:25:00) - Actual Fan State: False
Requesting FAN OFF again (should work now)...
Step 6 (2025-07-01T06:30:00) - Actual Fan State: False
Immediately Requesting FAN ON (expecting it to stay OFF - MIN_FAN_OFF_DURATION is 0:10:00)...
Step 7 (2025-07-01T06:35:00) - Actual Fan State: True
Running 2 more steps to pass MIN_FAN_OFF_DURATION...
Step 8 (2025-07-01T06:40:00) - Actual Fan State: True
Step 9 (2025-07-01T06:45:00) - Actual Fan